# Stereographic Encoding - Basic Symbolic Computations

This notebook demonstrates symbolic/analytical computations for stereographic projection
and quantum encoding using SymPy.

Based on theory from `ms/draft.pdf`.

In [ ]:
import sys
sys.path.append('../../src')

import sympy as sp
from sympy import symbols, sqrt, exp, I, simplify, expand
from sympy import init_printing, latex

# Initialize pretty printing
init_printing(use_latex='mathjax')

# Import our symbolic modules
from stereo_block_enc.symbolic.stereographic import StereographicEncoding
from stereo_block_enc.symbolic.mobius import PauliMobius, MobiusTransformation
from stereo_block_enc.symbolic.qsp import QSPStereographic

## 1. Stereographic Encoding

The stereographic encoding maps $z \in \mathbb{C}$ to $|z\rangle$:

$$|z\rangle = \frac{1}{\sqrt{|z|^2 + 1}}(z|0\rangle + |1\rangle)$$

In [ ]:
# Create encoding object
stereo = StereographicEncoding()

# Symbolic state
z = symbols('z', complex=True)
state = stereo.encoding_state(z)
print("Encoding state |z⟩:")
state

In [ ]:
# Polar form
r, phi = symbols('r phi', real=True)
state_polar = stereo.encoding_state_polar(r, phi)
print("In polar form z = r*exp(iφ):")
state_polar

## 2. Bloch Vector Representation

From theory (Eqs. 5-7):

$$\langle X \rangle = \frac{2\text{Re}(z)}{|z|^2 + 1}, \quad
\langle Y \rangle = \frac{2\text{Im}(z)}{|z|^2 + 1}, \quad
\langle Z \rangle = \frac{|z|^2 - 1}{|z|^2 + 1}$$

In [ ]:
# Bloch vector components
X, Y, Z = stereo.bloch_vector(z)
print("⟨X⟩ =", X)
print("⟨Y⟩ =", Y)
print("⟨Z⟩ =", Z)

### Decoding from Bloch vector (Eq. 20)

$$z = \frac{\langle X \rangle + i\langle Y \rangle}{1 - \langle Z \rangle}$$

In [ ]:
# Verify encoding-decoding roundtrip
z_decoded = stereo.decode_from_bloch(X, Y, Z)
z_decoded_simplified = simplify(z_decoded)
print("Decoded z:")
z_decoded_simplified

## 3. Möbius Transformations from Pauli Gates

From theory (Section 2.2):
- $X|z\rangle \to 1/z$
- $Y|z\rangle \to -1/z$
- $Z|z\rangle \to -z$
- $H|z\rangle \to (1+z)/(1-z)$

In [ ]:
# Pauli X transformation
X_mobius = PauliMobius.X()
print("X gate: z →", simplify(X_mobius(z)))

# Pauli Y transformation
Y_mobius = PauliMobius.Y()
print("Y gate: z →", simplify(Y_mobius(z)))

# Pauli Z transformation
Z_mobius = PauliMobius.Z()
print("Z gate: z →", simplify(Z_mobius(z)))

# Hadamard transformation
H_mobius = PauliMobius.H()
print("H gate: z →", simplify(H_mobius(z)))

### Composition of transformations

In [ ]:
# H^2 should be identity
H2 = H_mobius.compose(H_mobius)
result = simplify(H2(z))
print("H² applied to z =", result)

# X^2 should also be identity
X2 = X_mobius.compose(X_mobius)
result2 = simplify(X2(z))
print("X² applied to z =", result2)

## 4. Quantum Signal Processing

From Section 5, the encoding unitary is:

$$U_z = \frac{1}{\sqrt{1+r^2}}\begin{pmatrix}r e^{i\theta} & 1 \\ e^{-i\theta} & -r\end{pmatrix}$$

In [ ]:
# QSP object
qsp = QSPStereographic()

# Encoding unitary
theta = symbols('theta', real=True)
U_z = qsp.encoding_unitary(r, theta)
print("Encoding unitary U_z:")
U_z

In [ ]:
# Signal operator U_z σ_z
U_sigma = qsp.signal_operator(r, theta)
print("Signal operator U_z σ_z:")
U_sigma

### Chebyshev polynomials

For $\theta=0$ (real case), $(U_z \sigma_z)^k|0\rangle$ generates:

$$|\psi_k\rangle = T_k(\tilde{r})|0\rangle + \frac{U_{k-1}(\tilde{r})}{\sqrt{1+r^2}}|1\rangle$$

where $\tilde{r} = r/\sqrt{1+r^2}$

In [ ]:
# Compute states for k = 2, 3, 4
for k in [2, 3, 4]:
    print(f"\nk = {k}:")
    c0, c1 = qsp.qsp_state_coefficients(k, r)
    print(f"  |0⟩ coefficient: {c0}")
    print(f"  |1⟩ coefficient: {simplify(c1)}")

### Rational polynomials

Decoding via $z_k = (\langle X\rangle + i\langle Y\rangle)/(1-\langle Z\rangle)$ gives rational polynomials.

In [ ]:
# Rational polynomial for k=2
print("Rational polynomial for k=2:")
poly_2 = qsp.rational_polynomial(2, r)
simplify(poly_2)

In [ ]:
# For k=3
print("Rational polynomial for k=3:")
poly_3 = qsp.rational_polynomial(3, r)
simplify(poly_3)

## 5. Save formulas for later use

Export key formulas as LaTeX for the manuscript or as Python code for numerical implementation.

In [ ]:
# LaTeX export example
print("Encoding state in LaTeX:")
print(latex(state_polar))

print("\nBloch Z component in LaTeX:")
print(latex(Z))